# Tensor 

## Custom architecture of code for transformer of 200 million parameters

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


### Config

In [2]:
# Embedding dimension of transformer model
embedding_dim = 512
# feed forward network 
input_size = embedding_dim
hidden_size = embedding_dim * 4
output_size = embedding_dim

### Feed forward network  

In [3]:
# feedforward function for the custom model
class FeedForward(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(FeedForward, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.selu = torch.nn.SiLU()
        self.fc2 = torch.nn.Linear(hidden_size, hidden_size)
        self.selu = torch.nn.SiLU()
        self.fc3 = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x_1= self.fc1(x)
        out = self.selu(x_1)
        x_2 = self.fc2(out)
        out = self.selu(x_2)
        out = self.fc3(out)
        return out

# Layer Normolization 

### RMSNorm (Root Mean Square Normalization)

RMSNorm is a simpler version of LayerNorm. It only rescales each token's vector by its root mean square. There is no mean subtraction and no bias, so it is faster and has fewer parameters.

$$\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}$$

$$\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot g_i$$

where:
- $d$ = embedding dimension
- $g$ = learnable weight (one per feature)
- $\epsilon$ = small constant to avoid division by zero

Used in LLaMA, Mistral, Gemma and most modern LLMs.

In [ ]:
# using RMSNorm (Root Mean Square Layer Normalization) 
class layer_norm(nn.Module):
    def __init__(self,embedding_dim,eps = 1e-5):
        super(layer_norm,self).__init__()
        self.eps = eps
        self.embedding_dim = embedding_dim
        self.weight = nn.Parameter(torch.ones(embedding_dim))
        
    def forward(self,x):
        rms_norm = nn.RMSNorm(self.embedding_dim, eps=self.eps)
        return rms_norm(x) * self.weight
        

# Multi-Head Attention Block

In [ ]:
# MultiHeadAttention class for the custom model  written from scratch using nn.Module in PyTorch. It implements the multi-head attention mechanism used in transformer models. The class takes input dimensions, output dimensions, context length, number of heads, and an optional bias for query, key, and value projections. It computes the query, key, and value matrices, applies scaled dot-product attention with causal masking, and projects the output back to the desired dimension.

class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length, num_heads,qkv_bias=False):
        super().__init__()
        assert d_in % num_heads == 0, "d_in must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_out = d_out
        self.head_dim = d_in // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
            )
        
    def forward(self, x):
        batch_size, seq_length, _ = x.size()
        # Compute Q, K, V
        Q = self.W_query(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_key(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_value(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)

        """# Compute attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_scores = attn_scores.masked_fill(self.mask[:seq_length, :seq_length], float('-inf'))
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Compute output
        out = torch.matmul(attn_probs, V)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_out)
        out = self.out_proj(out)
        return out """
        
        out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_out)
        
        return self.out_proj(out)